# Optimización de Crop3

Se optimizan gamma y delta con igualdad de mezcla en cada iteración. Las pruebas usan las mismas mezclas de entrenamiento y semillas. Se maximiza SSIM y se valida sobre test con semillas independientes.


In [ ]:
import importlib
import sys
from pathlib import Path

import numpy as np
import tensorflow as tf
import optuna
import matplotlib.pyplot as plt

project_root = next(
    root for root in (Path.cwd(), *Path.cwd().parents)
    if (root / "project" / "CROP" / "models").is_dir()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from project.loader import Loader
from project.CROP.models.crop3 import Crop3
from project.CROP.models.crop2 import Crop2


SEED = 58642
#SEED = 3333

DATASET = "fashion"
INTER= 256
LAT=64 
CVAE_NAME = f"int_{INTER}_lat_{LAT}_{DATASET}.keras"
PREDICTOR_NAME = f"CCE_Conv2D_{DATASET}.keras"
N_IMAGES = 1000
N_TRIALS = 100
TARGET_METRIC = "acc_both"  # Maximizar el SSIM medio de las estimaciones.
SEPARATION_ITERATIONS = 30
assert TARGET_METRIC in {"acc_both", "ssim", "recon_bpsnr"}

FIXED_PARAMS = {
    "alpha_1": -2,
    "alpha_2": -22,
    "bias": 0.22,
    "slope": 22,
}

tf.keras.utils.set_random_seed(SEED)
data = Loader.data(dataset=DATASET)
cvae = Loader.cvae(model_name=CVAE_NAME)
predictor = Loader.predictor(model_name=PREDICTOR_NAME)

# Generador independiente de las semillas del CVAE y de Optuna.
rng = np.random.default_rng(SEED)
idx0 = rng.permutation(len(data["x_train"]))[:N_IMAGES]
idx1 = rng.permutation(len(data["x_train_1"]))[:N_IMAGES]
train_batch = (
    data["x_train"][idx0], data["x_train_1"][idx1],
    data["y_train"][idx0], data["y_train_1"][idx1],
)


def scalar_metric(metrics, key):
    value = metrics[key]
    # Las métricas de calidad devuelven (media, desvío).
    if isinstance(value, (tuple, list)):
        value = value[0]
    return float(value)


def objective(trial):
    params = {
        **FIXED_PARAMS,
        "gamma": trial.suggest_float("gamma", 0.0, 2.0),
        #"delta": trial.suggest_float("delta", 0.0, 1.0),
    }
    tf.keras.utils.set_random_seed(SEED)
    model = Crop2(cvae=cvae, predictor=predictor)
    metrics = model.unmix(
        *train_batch,
        params=params,
        iterations=SEPARATION_ITERATIONS,
        labels=data["labels"],
        show_image=False,
        show_metrics=False,
    )
    score = scalar_metric(metrics, TARGET_METRIC)
    if not np.isfinite(score):
        raise ValueError(f"Métrica no finita: {TARGET_METRIC}={score}")
    trial.set_user_attr("effective_params", params)
    for key in ("acc_both", "ssim", "recon_bpsnr"):
        trial.set_user_attr(key, scalar_metric(metrics, key))
    return score


Usando fashion como dataset


## Estudio

Se optimizan gamma y delta durante 20 iteraciones de separación. Cada prueba queda guardada en SQLite.


In [9]:
from datetime import datetime

output_dir = project_root / "project" / "experiments" / "optim_params" / "results"
output_dir.mkdir(parents=True, exist_ok=True)
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
study_path = output_dir / f"crop3_test2_{DATASET}_{TARGET_METRIC}_{run_id}.sqlite3"
study = optuna.create_study(
    direction="maximize",
    storage=f"sqlite:///{study_path}",
    study_name=f"Crop3Test2_{DATASET}_{TARGET_METRIC}",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
print("Estudio persistente:", study_path, flush=True)
study.optimize(objective, n_trials=N_TRIALS, n_jobs=1)
best_params = {**FIXED_PARAMS, **study.best_params}
print("Mejor valor:", study.best_value)
print("Parámetros completos:", best_params)


[I 2026-11-23 17:47:04,273] A new study created in RDB with name: Crop3Test2_fashion_acc_both


Estudio persistente: /home/galsan/Desktop/tesis/CROP_1/project/experiments/optim_params/results/crop3_test2_fashion_acc_both_20261123_174704.sqlite3


[I 2026-11-23 17:47:06,734] Trial 0 finished with value: 0.431 and parameters: {'gamma': 1.5096749894455816}. Best is trial 0 with value: 0.431.
[I 2026-11-23 17:47:09,361] Trial 1 finished with value: 0.351 and parameters: {'gamma': 1.6782476619447109}. Best is trial 0 with value: 0.431.
[I 2026-11-23 17:47:12,035] Trial 2 finished with value: 0.354 and parameters: {'gamma': 1.6690273807561191}. Best is trial 0 with value: 0.431.
[I 2026-11-23 17:47:14,510] Trial 3 finished with value: 0.563 and parameters: {'gamma': 0.9777384306971371}. Best is trial 3 with value: 0.563.
[I 2026-11-23 17:47:17,277] Trial 4 finished with value: 0.436 and parameters: {'gamma': 1.5039476354934498}. Best is trial 3 with value: 0.563.
[I 2026-11-23 17:47:20,455] Trial 5 finished with value: 0.615 and parameters: {'gamma': 0.05290418103556038}. Best is trial 5 with value: 0.615.
[I 2026-11-23 17:47:23,429] Trial 6 finished with value: 0.562 and parameters: {'gamma': 0.9841637115906414}. Best is trial 5 wit

Mejor valor: 0.64
Parámetros completos: {'alpha_1': -2, 'alpha_2': -22, 'bias': 0.22, 'slope': 22, 'gamma': 0.3123100836314193}


## Visualización de la optimización

Gráficos con Matplotlib: no requieren Plotly ni instalaciones dentro del notebook.

In [10]:
from optuna.visualization import plot_slice

plot_slice(study, params=list(study.best_params), target_name="Extitud de ambos objetos")



ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'marker': {'color': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,
                                   14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25,
                                   26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37,
                                   38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49,
                                   50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61,
                                   62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73,
                                   74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85,
                                   86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97,
                                   98, 99],
                         'colorbar': {'title': {'text': 'Trial'}, 'x': 1.0, 'xpad': 40},
                         'colorscale': [[0.0, 'rgb(247,251,255)'], [0.125,
                                        'rgb(222,235,247)'], [0.25,
                                        'rgb(198,219,239)'], [0.375,
                                        'rgb(158,202,225)'], [0.5,
                                        'rgb(107,174,214)'], [0.625,
                                        'rgb(66,146,198)'], [0.75,
                                        'rgb(33,113,181)'], [0.875,
                                        'rgb(8,81,156)'], [1.0, 'rgb(8,48,107)']],
                         'line': {'color': 'Grey', 'width': 0.5}},
              'mode': 'markers',
              'name': 'Feasible Trial',
              'showlegend': False,
              'type': 'scatter',
              'x': [1.5096749894455816, 1.6782476619447109, 1.6690273807561191,
                    0.9777384306971371, 1.5039476354934498, 0.05290418103556038,
                    0.9841637115906414, 0.8789135109972825, 1.8972473894029525,
                    0.7889067027644343, 0.012632854064797666, 0.112888048605851,
                    0.012676312082870789, 0.3639167422504459, 0.55766021421833,
                    0.4609101361086245, 0.41267871532866685, 0.4202820656501406,
                    0.6387899742769455, 1.2057383815306588, 0.2599858496372949,
                    0.3123100836314193, 0.24969732934125644, 0.2989008165311927,
                    0.2618170690705688, 0.684313530832629, 0.24873719993351556,
                    1.1187620607053417, 0.5092445081277601, 0.7503809272518633,
                    0.3367877750528218, 0.36460831910846203, 0.1749072737993699,
                    0.5984760382161651, 0.3730067461525322, 0.1718234639877543,
                    0.3563041581582057, 0.5167737574455996, 0.8967298994753831,
                    1.2336246216896298, 0.12537641480009698, 0.27791947857054833,
                    0.338347900518658, 0.7177169570927366, 0.10357465308185393,
                    0.32429370881130415, 1.939640968819411, 1.777788767747694,
                    0.4971706268318146, 1.4704213365020031, 0.8656588059778569,
                    0.33071433397620953, 0.30310130583876294, 0.20144640503015804,
                    0.0774770432022191, 0.4505425833712647, 0.31023501056480896,
                    0.4359338082489898, 0.5582897612436486, 0.03550371892256121,
                    0.6318812449455602, 0.30185835509788833, 0.2896279861944087,
                    0.1775706817274182, 0.21524446891863402, 0.4116066164309093,
                    0.29707230290270964, 0.15439805484287855, 0.2974380583560109,
                    0.00020112172002939577, 0.5555704872361646, 0.3040973355076305,
                    0.22812281102702214, 0.40112079598304845, 0.48781650109397656,
                    0.07878537160484739, 0.309162521425647, 0.39003693178916976,
                    0.1295780071410751, 0.22835581485333462, 0.45602267047390777,
                    0.31956276255307237, 0.32089275728487743, 0.3438666416791241,
                    0.26023206255834436, 0.5427313709409476, 0.31285573459740346,
                    0.603968828092251, 0.18812196797958003, 0.37816414259